In [1]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

# Model & processor
model_path = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"

model = LlavaForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_path)

def explainXray(classification_result, image_path):
  image = Image.open(image_path).convert("RGB")

  # Build prompt
  question_text = f"This brain MRI suggests ${classification_result}. What is the finding in the image that supports this diagnosis?"
  prompt = f"USER: <image>\n{question_text} ASSISTANT:"

  # Prepare input
  inputs = processor(
      images=[image],
      text=prompt,
      return_tensors="pt"
  ).to(model.device, torch.float16)

  # Generate output
  with torch.inference_mode():
      out = model.generate(
          **inputs,
          max_new_tokens=256,
          do_sample=False,
          temperature=0.0,
          pad_token_id=processor.tokenizer.eos_token_id
      )

  # Decode cleanly
  response = processor.decode(out[0], skip_special_tokens=True)
  print("\n--- MRI Findings ---\n")
  print(response.strip())

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [2]:
explainXray('glioma tumor', 'test_image_1.jpg')

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- MRI Findings ---

USER: 
This brain MRI suggests $glioma tumor. What is the finding in the image that supports this diagnosis? ASSISTANT: The image shows a mass in the left frontal region of the brain. This mass is associated with a midline shift, which is a displacement of the brain's central structures. The presence of a mass and midline shift are findings that can be indicative of a glioma tumor. However, it is important to note that further diagnostic tests and clinical correlation are needed to confirm the diagnosis and determine the appropriate course of treatment.


In [3]:
explainXray('meningioma tumor', 'test_image_2.jpg')


--- MRI Findings ---

USER: 
This brain MRI suggests $meningioma tumor. What is the finding in the image that supports this diagnosis? ASSISTANT: The image shows a large, well-defined, and homogenous mass in the left frontal region of the brain. This mass is causing a mass effect, which means it is exerting pressure on the surrounding brain tissue. Additionally, there is a midline shift, which indicates that the mass is displacing the brain's central structures. These findings are consistent with a meningioma tumor, which is a type of brain tumor that arises from the meninges, the protective layers surrounding the brain and spinal cord.


In [4]:
explainXray('no tumor', 'test_image_3.jpg')


--- MRI Findings ---

USER: 
This brain MRI suggests $no tumor. What is the finding in the image that supports this diagnosis? ASSISTANT: The image shows a normal brain MRI, which means that there are no visible abnormalities or tumors detected in the brain. This is a positive finding, as it indicates that the patient does not have a brain tumor.


In [5]:
explainXray('pituitary tumor', 'test_image_4.jpg')


--- MRI Findings ---

USER: 
This brain MRI suggests $pituitary tumor. What is the finding in the image that supports this diagnosis? ASSISTANT: The image shows a mass in the right sphenoid sinus, which is an abnormal growth or lesion. This finding, along with the patient's clinical presentation and other diagnostic tests, supports the diagnosis of a pituitary tumor.
